# Notebook: Apache Spark com Delta Lake

Notebook completo com modelagem ER, DDL, operações DML e time travel.

## 1) Delta Lake

Delta Lake fornece transações ACID e versionamento para dados em data lake.

In [50]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date
from delta import configure_spark_with_delta_pip

if 'spark' in globals():
    spark.stop()

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
data_path = project_root / 'data' / 'vendas.csv'
delta_path = project_root / 'data' / 'delta' / 'pedidos_delta'
warehouse_path = project_root / 'data' / 'delta' / 'warehouse'

if delta_path.exists():
    import shutil
    shutil.rmtree(delta_path)
if warehouse_path.exists():
    import shutil
    shutil.rmtree(warehouse_path)

builder = SparkSession.builder.appName('ProjetoDeltaLake').master('local[*]') \
 .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension') \
 .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog') \
 .config('spark.sql.warehouse.dir', str(warehouse_path.resolve())) \
 .config("spark.ui.port", "4041")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print('Spark:', spark.version)
print('CSV:', data_path)

Spark: 3.5.1
CSV: /home/dev-luiz/Documentos/GitHub/Apache-Spark/data/vendas.csv


## 2) Leitura do CSV

In [51]:
df_raw = spark.read.option('header', True).option('inferSchema', True).csv(str(data_path))
df = (df_raw
      .withColumn('data_pedido', to_date(col('data_pedido'), 'yyyy-MM-dd'))
      .withColumn('quantidade', col('quantidade').cast('int'))
      .withColumn('valor_unitario', col('valor_unitario').cast('double'))
      .withColumn('valor_total', col('valor_total').cast('double')))
df.printSchema()
df.show(5, truncate=False)

root
 |-- id_pedido: integer (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- cliente: string (nullable = true)
 |-- produto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- valor_unitario: double (nullable = true)
 |-- valor_total: double (nullable = true)
 |-- status: string (nullable = true)

+---------+-----------+----------------+------------------+--------------+----------+--------------+-----------+--------+
|id_pedido|data_pedido|cliente         |produto           |categoria     |quantidade|valor_unitario|valor_total|status  |
+---------+-----------+----------------+------------------+--------------+----------+--------------+-----------+--------+
|1        |2024-11-23 |Daniel Rocha    |Fone Bluetooth Pro|Eletrônicos   |2         |287.96        |575.92     |pendente|
|2        |2024-12-12 |Yasmin Oliveira |Secador Íon       |Beleza        |5         |248.89        |1244.45    |pago    |
|3        |2

## 3) Modelagem ER

```text
+-------------------------+
|        PEDIDOS          |
+-------------------------+
| id_pedido (PK)          |
| data_pedido             |
| cliente                 |
| produto                 |
| categoria               |
| quantidade              |
| valor_unitario          |
| valor_total             |
| status                  |
+-------------------------+
```

## 4) DDL

```sql
CREATE TABLE IF NOT EXISTS default.pedidos_delta (
    id_pedido INT,
    data_pedido DATE,
    cliente STRING,
    produto STRING,
    categoria STRING,
    quantidade INT,
    valor_unitario DOUBLE,
    valor_total DOUBLE,
    status STRING
) USING DELTA LOCATION '<path>';
```

In [52]:
delta_path_str = str(delta_path.resolve())
df.write.format('delta').mode('overwrite').save(delta_path_str)
spark.sql('DROP TABLE IF EXISTS default.pedidos_delta')
spark.sql("CREATE TABLE default.pedidos_delta USING DELTA LOCATION '" + delta_path_str + "'")
spark.sql('SELECT COUNT(*) AS total_registros FROM default.pedidos_delta').show()

[Stage 7:=============>                                          (12 + 12) / 50]

+---------------+
|total_registros|
+---------------+
|            100|
+---------------+



## 5) INSERT

In [53]:
spark.sql("INSERT INTO default.pedidos_delta VALUES (1001, DATE'2024-12-20', 'Renata Prado', 'Kindle 16GB', 'Eletrônicos', 1, 549.90, 549.90, 'pago')")
spark.sql("INSERT INTO default.pedidos_delta VALUES (1002, DATE'2024-12-21', 'Igor Tavares', 'Mochila Executiva', 'Roupas', 2, 189.90, 379.80, 'pendente')")
spark.sql('SELECT * FROM default.pedidos_delta WHERE id_pedido >= 1001').show(truncate=False)

+---------+-----------+------------+-----------------+-----------+----------+--------------+-----------+--------+
|id_pedido|data_pedido|cliente     |produto          |categoria  |quantidade|valor_unitario|valor_total|status  |
+---------+-----------+------------+-----------------+-----------+----------+--------------+-----------+--------+
|1002     |2024-12-21 |Igor Tavares|Mochila Executiva|Roupas     |2         |189.9         |379.8      |pendente|
|1001     |2024-12-20 |Renata Prado|Kindle 16GB      |Eletrônicos|1         |549.9         |549.9      |pago    |
+---------+-----------+------------+-----------------+-----------+----------+--------------+-----------+--------+



## 6) UPDATE

In [54]:
spark.sql("UPDATE default.pedidos_delta SET status = 'pago' WHERE id_pedido = 1002")
spark.sql('SELECT id_pedido, cliente, status FROM default.pedidos_delta WHERE id_pedido = 1002').show()

+---------+------------+------+
|id_pedido|     cliente|status|
+---------+------------+------+
|     1002|Igor Tavares|  pago|
+---------+------------+------+



## 7) DELETE

In [55]:
antes = spark.sql("SELECT COUNT(*) AS qtd FROM default.pedidos_delta WHERE status = 'cancelado' AND id_pedido <= 100").collect()[0]['qtd']
spark.sql("DELETE FROM default.pedidos_delta WHERE status = 'cancelado' AND id_pedido <= 100")
depois = spark.sql("SELECT COUNT(*) AS qtd FROM default.pedidos_delta WHERE status = 'cancelado' AND id_pedido <= 100").collect()[0]['qtd']
print('Cancelados antes:', antes, '| depois:', depois)

Cancelados antes: 17 | depois: 0


## 8) SELECT com filtros

In [56]:
spark.sql("SELECT categoria, status, COUNT(*) AS pedidos, ROUND(SUM(valor_total), 2) AS faturamento FROM default.pedidos_delta WHERE data_pedido BETWEEN DATE'2024-01-01' AND DATE'2024-12-31' GROUP BY categoria, status ORDER BY faturamento DESC").show(50, truncate=False)

+--------------+--------+-------+-----------+
|categoria     |status  |pedidos|faturamento|
+--------------+--------+-------+-----------+
|Eletrônicos   |pago    |14     |22286.8    |
|Eletrônicos   |pendente|5      |12808.65   |
|Casa e Cozinha|pago    |10     |11451.16   |
|Beleza        |pago    |14     |7770.09    |
|Casa e Cozinha|pendente|7      |5952.83    |
|Livros        |pago    |15     |4606.98    |
|Roupas        |pago    |11     |4186.76    |
|Roupas        |pendente|4      |2043.06    |
|Beleza        |pendente|3      |1115.14    |
|Livros        |pendente|2      |490.41     |
+--------------+--------+-------+-----------+



## 9) Time Travel / Versionamento

In [57]:
hist = spark.sql('DESCRIBE HISTORY default.pedidos_delta')
hist.select('version', 'timestamp', 'operation').show(truncate=False)
versao0 = spark.read.format('delta').option('versionAsOf', 0).load(delta_path_str)
print('Registros na versão 0:', versao0.count())
versao0.show(5, truncate=False)

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|4      |2026-04-28 20:44:39.647|DELETE   |
|3      |2026-04-28 20:44:35.232|UPDATE   |
|2      |2026-04-28 20:44:31.399|WRITE    |
|1      |2026-04-28 20:44:30.987|WRITE    |
|0      |2026-04-28 20:44:28.501|WRITE    |
+-------+-----------------------+---------+

Registros na versão 0: 100
+---------+-----------+----------------+------------------+--------------+----------+--------------+-----------+--------+
|id_pedido|data_pedido|cliente         |produto           |categoria     |quantidade|valor_unitario|valor_total|status  |
+---------+-----------+----------------+------------------+--------------+----------+--------------+-----------+--------+
|1        |2024-11-23 |Daniel Rocha    |Fone Bluetooth Pro|Eletrônicos   |2         |287.96        |575.92     |pendente|
|2        |2024-12-12 |Yasmin Oliveira |Secador Íon       |Beleza        |5         |248

## 10) ACID

- Atomicidade
- Consistência
- Isolamento
- Durabilidade

In [62]:
spark.stop()
print("-"*40)
print('Sessão Spark encerrada.')
print("-"*40)

----------------------------------------
Sessão Spark encerrada.
----------------------------------------
